# **TP1 - Variable Types & Descriptive Statistics**

<b>Exploratory Data Analysis & Unsuperivsed Learning </b><br>
**Course: Dr. HAS Sothea** <br> 
**TP: Ms. UANN Sreyvi** <br>

-------

**Objective**: This initial practical session is designed to enhance your understanding of various data variable types and the corresponding statistical and graphical tools suitable for each type.

---------

> **The `Jupyter Notebook` for this TP can be downloaded here: [TP1-Gapminder](https://hassothea.github.io/I5_EDA_ITC/TPs/TP1_Variable_Types_Descriptive_Statistics.ipynb)**.

-------

# 1. `Gapminder` dataset

[`Gapminder`](https://pypi.org/project/gapminder/) is an excerpt of data available at [https://gapminder.org](https://www.gapminder.org/). For each of $142$ countries (`country`), the package provides values for life expectancy (`lifeExp`), GDP per capita (`gdpPercap`), and population (`pop`), every five years, from $1952$ to $2007$ (`year`). It is initially used in [Jennifer Bryan's excellent `gapminder` teaching package for R](https://github.com/jennybc/gapminder/) ([`ggplot`](https://ggplot2.tidyverse.org/), [`tidyverse`](https://www.tidyverse.org/), and more). For more information about `gapminder`: 
- Documentation: [https://www.gapminder.org/data/documentation/](https://www.gapminder.org/data/documentation/)
- A short demonstration video is available here: [Hans Rosling's 200 Countries, 200 Years, 4 Minutes - The Joy of Stats - BBC Four](https://youtu.be/jbkSRLYSojo?si=qipg08VIi999hEgo).

In [2]:
# useful packages
import numpy as np
import pandas as pd

# %pip install gapminder  #(for installing gapminder if you haven't had one)
from gapminder import gapminder
print(f"* Number of observations: {gapminder.shape[0]}")
gapminder.sample(3)

* Number of observations: 1704


,country,continent,year,lifeExp,pop,gdpPercap
858,Kuwait,Asia,1982,71.309,1497494,31354.035730
901,Libya,Africa,1957,45.289,1201578,3448.284395
1099,New Zealand,Oceania,1987,74.320,3317166,19007.191290


----------

# 2. Variable types

`EDA` involves summarizing and visualizing data to uncover patterns, detect anomalies, and understand relationships between variables. Statistical summaries, such as mean, median, and standard deviation, are essential tools in this process.

- Which variables are considered quantitative and which are qualitative? 

> Hint: You can check the default column types by using `gapminder.dtypes`. But this may not be accurate because some categorical data may be encoded using numerical values.

In [3]:
gapminder.dtypes

country          str
continent        str
year           int64
lifeExp      float64
pop            int64
gdpPercap    float64
dtype: object

In [4]:
print(f"Quantitative columns: {gapminder.select_dtypes(include='number').columns.to_list()}")
print(f"Qualitative columns: {gapminder.select_dtypes(include='object').columns.to_list()}")

Quantitative columns: ['year', 'lifeExp', 'pop', 'gdpPercap']
Qualitative columns: ['country', 'continent']


C:\Users\Meng\AppData\Local\Temp\ipykernel_14520\4108785752.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(f"Qualitative columns: {gapminder.select_dtypes(include='object').columns.to_list()}")


> **Your response**: 

## 2.1 Quantitative variables

1. In year 2002, 
- Compute suitable statistics for each quantitative variable (excluding `year`) to obtain an overall summary.
- Recall the definitions of **skewness** and **kurtosis** introduced in the course. 
- Compute these metrics for each quantitative variable and explain the distribution of each variable based on these values.

In [5]:
data2002 = gapminder.query("year == 2002")
data2002.head()

,country,continent,year,lifeExp,pop,gdpPercap
10,Afghanistan,Asia,2002,42.129,25268405,726.734055
22,Albania,Europe,2002,75.651,3508512,4604.211737
34,Algeria,Africa,2002,70.994,31287142,5288.040382
46,Angola,Africa,2002,41.003,10866106,2773.287312
58,Argentina,Americas,2002,74.340,38331121,8797.640716


In [6]:
data2002[['lifeExp', 'pop',	'gdpPercap']].describe()

,lifeExp,pop,gdpPercap
count,142.000000,1.420000e+02,142.000000
mean,65.694923,4.145759e+07,9917.848365
std,12.279823,1.408483e+08,11154.114865
min,39.193000,1.703720e+05,241.165876
25%,55.522250,4.173506e+06,1409.567264
50%,70.825500,1.037292e+07,5319.804524
75%,75.459250,2.654556e+07,13359.512258
max,82.000000,1.280400e+09,44683.975250


2. Graphically represent the distribution of each variable for the year $2002$. After plotting the distributions, provide a brief explanation for each variable.

In [7]:
import plotly.express as px
fig = px.histogram(
    data2002,
    x = 'lifeExp'
)
# config
fig.update_layout(
    width=500,
    height=300,
    title='Histogram of life expectancy in 2002'
)
fig.show()

3. According to the data, in 2002:
- Which country is the richest? 
- Which country is the poorest?
- Which country is the healthiest?
- Which country is the unhealthiest?

In [8]:
data2002.query("gdpPercap == gdpPercap.max()")

,country,continent,year,lifeExp,pop,gdpPercap
1150,Norway,Europe,2002,79.05,4535591,44683.97525


4. Repeat the previous question for the year 1977. Before computing, can you guess which country had the lowest life expectancy around that year?

In [9]:
gapminder.query("year == 1977").query("lifeExp == lifeExp.min()")

,country,continent,year,lifeExp,pop,gdpPercap
221,Cambodia,Asia,1977,31.22,6978607,524.972183


## 2.2. Qualitative variables

Qualitative variables are simpler than the quantitative ones, as we primarily focus on the proportion or frequency of each category. In our dataset, the existing qualitative variables are not suitable for analysis because they are repeated each year. Therefore, we will create three new qualitative variables associated with the three quantitative columns by copping them into three categories each.

In year 2002,

1. Add the following three variables to the `gapminder` dataset by grouping each quantitative variable into $3$ groups.
- Create variable `gdpQual` with three categories: [**"developing"**, **"moderate"**, **"developed"**] using variable `gdpPercap`.
- Create variable `popQual` with three categories: [**"small"**, **"medium"**, **"large"**] using variable `pop`.
- Create variable `lifeExpQual` with three categories: [**"unhealthy"**, **"moderate"**, **"healthy"**] using variable `lifeExp`.

> Hint: you may find `pd.cut` function helpful.

In [10]:
data2002['gdpQual'] = pd.cut(
    data2002['gdpPercap'],
    bins=3,
    labels=["developing", "moderate", "developed"])

In [11]:
data2002.head()

,country,continent,year,lifeExp,pop,gdpPercap,gdpQual
10,Afghanistan,Asia,2002,42.129,25268405,726.734055,developing
22,Albania,Europe,2002,75.651,3508512,4604.211737,developing
34,Algeria,Africa,2002,70.994,31287142,5288.040382,developing
46,Angola,Africa,2002,41.003,10866106,2773.287312,developing
58,Argentina,Americas,2002,74.340,38331121,8797.640716,developing


2. Compute the appropriate statistical values and graphically represent the distribution of each newly created qualitative variable.

In [12]:
gdp_qual = data2002.gdpQual.value_counts(normalize=True)
gdp_qual

gdpQual
developing    0.760563
moderate      0.133803
developed     0.105634
Name: proportion, dtype: float64

In [13]:
data2002['continent'].value_counts()

continent
Africa      52
Asia        33
Europe      30
Americas    25
Oceania      2
Name: count, dtype: int64

- Graphically represent them and describe the graph.

In [14]:
fig1 = px.bar(
    x = gdp_qual.index,
    y = gdp_qual.values
)
fig1.update_layout(
    width=500,
    height=400,
    title='Bar chart of qualitative GDP per capita in 2002'
)
fig1.show()

---------

# 3. Time evolution

`Gapminder` captures global changes from $1952$ to $2007$. It is more insightful to examine how these variables evolve over time.

## 3.1. Evolution of quantitative columns

1. Create a line plot of `lifeExp` for the five continents from $1952$ to $2007$ using `sns.lineplot`. What observations can you make from the plot?

2. Repeat the same process with `pop`, and then with `gdpPercap`. Provide your comments for each case.

In [15]:
# To do

3. Plot the evolution of each of the three quantitative variables for **Cambodia** vs **Thailand**. Describe the graphs.

In [16]:
# To do

## 3.2. Evolution of qualitative columns

1. Visualize the evolution of `lifeExpQual` column in **Asia** from 1952 to 2007. What do you observe?

In [17]:
# To do

2. Do the same with `popQual` and `dgpQual` columns.

In [18]:
# To do

3. Visualize the evolution of the three qualitative columns above for African countries from 1952 to 2007.

# Further readings
- Gapminder documentation: [https://www.gapminder.org/data/documentation/](https://www.gapminder.org/data/documentation/)
- A short demonstration video is available here: [Hans Rosling's 200 Countries, 200 Years, 4 Minutes - The Joy of Stats - BBC Four](https://youtu.be/jbkSRLYSojo?si=qipg08VIi999hEgo).
- Graphical tools:
    - [`matplotlib`](https://matplotlib.org/stable/index.html)
    - [`seaborn`](https://seaborn.pydata.org/)
    - [`https://plotly.com/python/`](https://plotly.com/python/)